# Option 2 — Symbolic Conditioned MIDI Generation

**Task**: Given a 4-second MIDI prefix, autoregressively generate a 4-second continuation using a small causal Transformer trained on MAESTRO v3.0.0.

**Pipeline**: Setup → Download MAESTRO → Pre-cache piano-rolls → Build windows → Train → Generate → Evaluate

## 1. Environment Setup

In [1]:
import os, sys, importlib
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
print('Running on Colab:', IN_COLAB)

if IN_COLAB:
    REPO_URL = 'https://github.com/archerop/CSE_253.git'
    REPO_DIR = Path('/content/CSE_253')
    if not REPO_DIR.exists():
        !git clone {REPO_URL} {REPO_DIR}
    else:
        result = !git -C {REPO_DIR} pull --ff-only
        print('\n'.join(result))
        # Flush stale cached modules so re-imports read the updated files
        stale = [k for k in sys.modules if k == 'app' or k.startswith('app.')]
        for k in stale:
            del sys.modules[k]
        importlib.invalidate_caches()
        if stale:
            print(f'Flushed {len(stale)} cached module(s)')
    ASSIGNMENT_ROOT = REPO_DIR / 'assignment2'
else:
    ASSIGNMENT_ROOT = Path('__file__').resolve().parent if '__file__' in dir() else Path('.').resolve()
    for p in [ASSIGNMENT_ROOT] + list(ASSIGNMENT_ROOT.parents):
        if (p / 'app').exists():
            ASSIGNMENT_ROOT = p
            break

print('Assignment root:', ASSIGNMENT_ROOT)
os.chdir(ASSIGNMENT_ROOT)
if str(ASSIGNMENT_ROOT) not in sys.path:
    sys.path.insert(0, str(ASSIGNMENT_ROOT))

Running on Colab: True
Cloning into '/content/CSE_253'...
remote: Enumerating objects: 64, done.
remote: Counting objects: 100% (64/64), done.
remote: Compressing objects: 100% (49/49), done.
remote: Total 64 (delta 11), reused 64 (delta 11), pack-reused 0 (from 0)
Receiving objects: 100% (64/64), 126.84 KiB | 6.68 MiB/s, done.
Resolving deltas: 100% (11/11), done.
Assignment root: /content/CSE_253/assignment2


## 2. Install Dependencies

In [2]:
import importlib, subprocess

def need(pkg):
    return importlib.util.find_spec(pkg) is None

pkgs = [p for p in ['pretty_midi', 'torch', 'pandas', 'tqdm', 'scipy'] if need(p)]
if pkgs:
    print('Installing:', pkgs)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + pkgs)
else:
    print('All dependencies already installed.')

Installing: ['pretty_midi']


## 3. Download MAESTRO MIDI Dataset

In [3]:
import urllib.request, zipfile

MAESTRO_ROOT = ASSIGNMENT_ROOT / 'data' / 'maestro-v3.0.0'

if (MAESTRO_ROOT / 'maestro-v3.0.0.csv').exists():
    print('MAESTRO already present at', MAESTRO_ROOT)
else:
    URL = 'https://storage.googleapis.com/magentadata/datasets/maestro/v3.0.0/maestro-v3.0.0-midi.zip'
    ZIP = ASSIGNMENT_ROOT / 'data' / 'downloads' / 'maestro-v3.0.0-midi.zip'
    ZIP.parent.mkdir(parents=True, exist_ok=True)
    if not ZIP.exists():
        print('Downloading MAESTRO MIDI (~57 MB)...')
        urllib.request.urlretrieve(URL, ZIP)
    print('Extracting...')
    with zipfile.ZipFile(ZIP) as zf:
        zf.extractall(ASSIGNMENT_ROOT / 'data')
    print('Done.')

print(f'MIDI files: {len(list(MAESTRO_ROOT.rglob("*.midi")))}')

Extracting...
Done.
MIDI files: 1276


## 4. Imports

In [4]:
import math, json, pickle
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import pretty_midi
from tqdm.auto import tqdm

from app.shared.config import (
    MAESTRO_ROOT, MIDI_LOW, MIDI_HIGH, N_PITCHES,
    OPTION2_CACHE_DIR, OPTION2_OUTPUT_DIR, CHECKPOINT_DIR,
    OPTION2_FRAME_RATE, OPTION2_PREFIX_SECONDS, OPTION2_CONTINUATION_SECONDS,
    OPTION2_STRIDE_SECONDS, OPTION2_D_MODEL, OPTION2_NHEAD, OPTION2_NUM_LAYERS,
    OPTION2_DIM_FEEDFORWARD, OPTION2_DROPOUT, OPTION2_BATCH_SIZE,
    OPTION2_LEARNING_RATE, OPTION2_WEIGHT_DECAY, OPTION2_MAX_EPOCHS, OPTION2_PATIENCE,
)
from app.shared.metadata import load_maestro_metadata, validate_maestro_paths
from app.option2.symbolic_dataset import (
    build_window_index, SymbolicDataset, precache_pianorolls,
    _midi_to_pianoroll, _load_roll,
)
from app.option2.symbolic_models import SymbolicTransformer, CopyLastFrameBaseline
from app.option2.symbolic_train import train, load_best_checkpoint
from app.option2.symbolic_generate import (
    extract_prefix, generate_conditioned, save_symbolic_conditioned, pianoroll_to_midi
)
from app.option2.symbolic_eval import evaluate_generation, print_metrics

print('Imports OK')

Imports OK


In [5]:
if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
elif torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
else:
    DEVICE = torch.device('cpu')
print('Device:', DEVICE)

Device: cuda


## 5. Google Drive Setup (Colab only)
Mounts Google Drive and routes the checkpoint there so it survives session resets.

In [6]:
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    GDRIVE_DIR = Path('/content/drive/MyDrive/CSE253')
    GDRIVE_DIR.mkdir(parents=True, exist_ok=True)
    CKPT_PATH = GDRIVE_DIR / 'option2_best.pt'
    print('Checkpoint → Google Drive:', CKPT_PATH)
else:
    CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
    CKPT_PATH = CHECKPOINT_DIR / 'option2_best.pt'
    print('Checkpoint → local:', CKPT_PATH)

Mounted at /content/drive
Checkpoint → Google Drive: /content/drive/MyDrive/CSE253/option2_best.pt


## 6. Pre-cache Piano-rolls
Parses all 1,276 MIDI files once and saves as `.npy`. Safe to re-run — skips existing files.

In [7]:
precache_pianorolls()

Caching 1276 piano-rolls to /content/CSE_253/assignment2/cache/option2/pianorolls ...


100%|██████████| 1276/1276 [04:44<00:00,  4.49file/s]

Pre-caching complete.


## 7. Build Window Index

In [8]:
OPTION2_CACHE_DIR.mkdir(parents=True, exist_ok=True)

train_windows = build_window_index('train',      cache_path=OPTION2_CACHE_DIR / 'train_windows.pkl')
val_windows   = build_window_index('validation', cache_path=OPTION2_CACHE_DIR / 'val_windows.pkl')
test_windows  = build_window_index('test',       cache_path=OPTION2_CACHE_DIR / 'test_windows.pkl')

print(f'Windows — train: {len(train_windows):,}  val: {len(val_windows):,}  test: {len(test_windows):,}')

Indexing [test]: 100%|██████████| 177/177 [00:00<00:00, 9995.98it/s]

Windows — train: 283,276  val: 34,493  test: 35,348


## 8. Dataset & DataLoader

In [9]:
train_ds = SymbolicDataset(train_windows)
val_ds   = SymbolicDataset(val_windows)

prefix, cont = train_ds[0]
print(f'prefix: {tuple(prefix.shape)}  cont: {tuple(cont.shape)}')
print(f'avg active notes/frame — prefix: {prefix.sum(-1).mean():.2f}  cont: {cont.sum(-1).mean():.2f}')

prefix: (160, 88)  cont: (160, 88)
avg active notes/frame — prefix: 1.58  cont: 2.61


In [10]:
PIN_MEMORY  = DEVICE.type == 'cuda'
NUM_WORKERS = 2 if IN_COLAB else 4

train_loader = DataLoader(train_ds, batch_size=OPTION2_BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
val_loader   = DataLoader(val_ds,   batch_size=OPTION2_BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

print(f'Train batches: {len(train_loader):,}  Val batches: {len(val_loader):,}')

Train batches: 8,853  Val batches: 1,078


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


## 9. Model

| Component | Value |
|-----------|-------|
| Architecture | Causal Transformer (GPT-style) |
| Input | 88-dim binary piano-roll frame |
| d_model | 128 |
| Layers | 4 |
| Heads | 4 |
| FFN dim | 512 |
| Loss | BCEWithLogitsLoss |
| Training | Teacher forcing over `[prefix \| continuation[:-1]]` |

In [11]:
model = SymbolicTransformer().to(DEVICE)
print(f'Parameters: {model.count_parameters():,}')

/content/CSE_253/assignment2/app/option2/symbolic_models.py:83: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)


Parameters: 815,832


## 10. Training

In [ ]:
history = train(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    device=DEVICE,
    checkpoint_path=CKPT_PATH,
)

OPTION2_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
with open(OPTION2_OUTPUT_DIR / 'train_history.json', 'w') as f:
    json.dump(history, f, indent=2)

print('Training complete. Checkpoint saved to:', CKPT_PATH)

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


Epoch   1/20 | train=0.0342  val=0.0273  best=0.0273 ✓
Epoch   2/20 | train=0.0315  val=0.0269  best=0.0269 ✓
Epoch   3/20 | train=0.0314  val=0.0270  best=0.0269 
Epoch   4/20 | train=0.0313  val=0.0269  best=0.0269 
Epoch   5/20 | train=0.0312  val=0.0270  best=0.0269 
Epoch   6/20 | train=0.0310  val=0.0266  best=0.0266 ✓
Epoch   7/20 | train=0.0309  val=0.0266  best=0.0266 
Epoch   8/20 | train=0.0309  val=0.0266  best=0.0266 
Epoch   9/20 | train=0.0309  val=0.0265  best=0.0265 ✓
Epoch  10/20 | train=0.0309  val=0.0265  best=0.0265 ✓
Epoch  11/20 | train=0.0309  val=0.0265  best=0.0265 
Epoch  12/20 | train=0.0309  val=0.0265  best=0.0265 
Epoch  13/20 | train=0.0309  val=0.0266  best=0.0265 
Epoch  14/20 | train=0.0308  val=0.0264  best=0.0264 ✓


## 11. Loss Curves

In [ ]:
import matplotlib.pyplot as plt

epochs = range(1, len(history['train_loss']) + 1)
plt.figure(figsize=(8, 4))
plt.plot(epochs, history['train_loss'], label='Train')
plt.plot(epochs, history['val_loss'],   label='Val')
plt.xlabel('Epoch')
plt.ylabel('BCE Loss')
plt.title('Option 2 Training Loss')
plt.legend()
plt.tight_layout()
plt.savefig(OPTION2_OUTPUT_DIR / 'loss_curve.png', dpi=150)
plt.show()

## 12. Generate `symbolic_conditioned.mid`

In [ ]:
df = load_maestro_metadata(MAESTRO_ROOT)
df = validate_maestro_paths(df)
test_df = df[(df['split'] == 'test') & df['midi_exists']].sort_values('duration').reset_index(drop=True)
row = test_df.iloc[len(test_df) // 2]
MIDI_PATH = row['midi_path']
print(f"Using: {row['composer']} — {row['title']}  ({row['duration']:.1f}s)")

In [ ]:
model = load_best_checkpoint(SymbolicTransformer(), CKPT_PATH, DEVICE)

output_path = save_symbolic_conditioned(
    prefix_midi_path=MIDI_PATH,
    model=model,
    device=DEVICE,
)
print('Output saved to:', output_path)

## 13. Evaluation — Model vs Baseline

In [ ]:
prefix_len = int(OPTION2_PREFIX_SECONDS * OPTION2_FRAME_RATE)
cont_len   = int(OPTION2_CONTINUATION_SECONDS * OPTION2_FRAME_RATE)

roll          = _load_roll(MIDI_PATH, OPTION2_FRAME_RATE)
gt_roll       = roll[prefix_len : prefix_len + cont_len]
prefix_tensor = extract_prefix(MIDI_PATH)

model_roll    = generate_conditioned(model, prefix_tensor, cont_len, DEVICE)
baseline_roll = CopyLastFrameBaseline()(prefix_tensor.unsqueeze(0), cont_len).squeeze(0).numpy()

print('=== Model ===')
print_metrics(evaluate_generation(model_roll, gt_roll))

print('=== Baseline (copy last frame) ===')
print_metrics(evaluate_generation(baseline_roll, gt_roll))

In [ ]:
import matplotlib.pyplot as plt

prefix_np = extract_prefix(MIDI_PATH).numpy()
combined  = np.concatenate([prefix_np, model_roll], axis=0).T

fig, ax = plt.subplots(figsize=(14, 4))
ax.imshow(combined, aspect='auto', origin='lower', cmap='Blues', interpolation='nearest')
ax.axvline(prefix_len - 0.5, color='red', linewidth=1.5, label='prefix | generated')
ax.set_xlabel('Frame (40 fps)')
ax.set_ylabel('Pitch (MIDI 21–108)')
ax.set_title('Piano-roll: prefix (left) + generated continuation (right)')
ax.legend(loc='upper right')
plt.tight_layout()
plt.savefig(OPTION2_OUTPUT_DIR / 'pianoroll_visualization.png', dpi=150)
plt.show()

## Done

| Output | Location |
|--------|----------|
| Generated MIDI | `outputs/option2/symbolic_conditioned.mid` |
| Best checkpoint | Google Drive `CSE253/option2_best.pt` (Colab) or `outputs/checkpoints/option2_best.pt` (local) |
| Loss curve | `outputs/option2/loss_curve.png` |
| Piano-roll plot | `outputs/option2/pianoroll_visualization.png` |